# 🎵 Song Popularity Predictor - Siklus 4 Enhanced

## Overview
Sistem prediksi popularitas lagu menggunakan Machine Learning dengan:
- **Feature Engineering**: Artist, audio, temporal, lyrics features
- **NLP Processing**: TF-IDF + Dimensionality Reduction (SVD)
- **Model**: LightGBM Regression dengan 5-Fold Cross-Validation
- **Analysis**: 15 comprehensive visualizations + detailed insights

**Author**: Tim AhThatsHot  
**Version**: 4.0 Enhanced

## 📦 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from pathlib import Path
import warnings
import re
warnings.filterwarnings('ignore')

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Machine Learning
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor

# Statistics
from scipy import stats

# Others
import joblib
from datetime import datetime

# Konfigurasi visualisasi
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## 🏗️ 2. SongPopularityPredictor Class Definition

Class ini berisi semua logic untuk:
- Data loading & EDA
- Feature engineering
- NLP processing
- Model training
- Analysis & visualization

In [ ]:
class SongPopularityPredictor:
    """
    SIKLUS 4 ENHANCED: Model Prediksi Popularitas Lagu

    Pipeline lengkap untuk memprediksi popularitas lagu berdasarkan:
    1. Fitur Audio (energy, danceability, tempo, dll)
    2. Fitur Artist (popularitas rata-rata, jumlah lagu)
    3. Fitur Temporal (tahun rilis, dekade, age)
    4. Fitur Lyrics (NLP dengan TF-IDF + SVD)
    5. Fitur Genre dan interaksi antar fitur
    """

    def __init__(self, data_path='.', output_path='./outputs'):
        """
        Inisialisasi predictor

        Args:
            data_path (str): Path folder yang berisi train.csv dan test.csv
            output_path (str): Path folder untuk menyimpan output
        """
        self.data_path = Path(data_path)
        self.output_path = Path(output_path)

        # Data containers
        self.train_df = None
        self.test_df = None

        # Features & models
        self.features = []
        self.models = {}
        self.scalers = {}

        # Preprocessing objects
        self.imputer = None
        self.label_encoders = {}
        self.categorical_features_raw = []
        self.categorical_features_encoded = []

        # Results
        self.model_results = None
        self.oof_predictions = None
        self.cv_scores = None

        # Buat output folder jika belum ada
        self.output_path.mkdir(parents=True, exist_ok=True)

    def load_data(self):
        """Load training dan testing data dari CSV files"""
        print("=" * 80)
        print("📂 LOADING DATA")
        print("=" * 80)

        self.train_df = pd.read_csv(self.data_path / 'train.csv', engine='python')
        self.test_df = pd.read_csv(self.data_path / 'test.csv', engine='python')

        print(f"✓ Training data: {self.train_df.shape}")
        print(f"✓ Testing data: {self.test_df.shape}")
        print(f"✓ Target range: [{self.train_df['popularity'].min():.0f}, {self.train_df['popularity'].max():.0f}]")
        print(f"✓ Target mean: {self.train_df['popularity'].mean():.2f}")
        print(f"✓ Target std: {self.train_df['popularity'].std():.2f}")

        return self

    def eda(self):
        """Exploratory Data Analysis - analisis statistik dasar"""
        print("\n" + "=" * 80)
        print("📊 EXPLORATORY DATA ANALYSIS")
        print("=" * 80)

        print("\nBasic Statistics:")
        print(self.train_df['popularity'].describe())

        print(f"\nMissing Values:")
        missing = self.train_df.isnull().sum()
        if missing.sum() > 0:
            print(missing[missing > 0])
        else:
            print("  No missing values in main features")

        print(f"\nTop 5 Genres:")
        print(self.train_df['track_genre'].value_counts().head())

        return self

    def engineer_features(self):
        """Feature Engineering - membuat fitur-fitur baru yang informatif"""
        print("\n" + "=" * 80)
        print("🔧 FEATURE ENGINEERING")
        print("=" * 80)

        # ============ 1. ARTIST FEATURES (Target Encoding) ============
        print("[1/7] Artist features (target encoding)...")

        # Hitung rata-rata popularitas per artist dari training data
        artist_popularity_map = self.train_df.groupby('artists')['popularity'].mean()
        artist_song_count_map = self.train_df.groupby('artists').size()

        # Global mean sebagai fallback untuk artist yang tidak dikenal
        global_mean_pop = self.train_df['popularity'].mean()
        global_mean_count = self.train_df['artists'].value_counts().mean()

        # Apply ke train dan test
        self.train_df['artist_avg_pop'] = self.train_df['artists'].map(artist_popularity_map)
        self.test_df['artist_avg_pop'] = self.test_df['artists'].map(artist_popularity_map)
        self.train_df['artist_avg_pop'] = self.train_df['artist_avg_pop'].fillna(global_mean_pop)
        self.test_df['artist_avg_pop'] = self.test_df['artist_avg_pop'].fillna(global_mean_pop)

        self.train_df['artist_song_count'] = self.train_df['artists'].map(artist_song_count_map)
        self.test_df['artist_song_count'] = self.test_df['artists'].map(artist_song_count_map)
        self.train_df['artist_song_count'] = self.train_df['artist_song_count'].fillna(global_mean_count)
        self.test_df['artist_song_count'] = self.test_df['artist_song_count'].fillna(global_mean_count)

        print("   ✓ artist_avg_pop, artist_song_count")

        # ============ 2-5. FITUR LAINNYA ============
        for df, name in [(self.train_df, 'Train'), (self.test_df, 'Test')]:

            # 2. Audio Features
            print(f"[2/7] Audio features ({name})...")
            if all(col in df.columns for col in ['energy', 'danceability']):
                df['energy_x_dance'] = df['energy'] * df['danceability']
            if 'duration_ms' in df.columns:
                df['duration_min'] = df['duration_ms'] / 60000
            if all(col in df.columns for col in ['key', 'mode']):
                df['key_mode'] = df['key'].astype(str) + '_' + df['mode'].astype(str)
            if 'tempo' in df.columns:
                df['tempo_category'] = pd.cut(df['tempo'],
                                              bins=[0, 90, 120, 150, 250],
                                              labels=['slow', 'moderate', 'fast', 'very_fast'])

            # 3. Temporal Features
            print(f"[3/7] Temporal features ({name})...")
            if 'release_year' in df.columns:
                df['years_since_release'] = 2025 - df['release_year']
                df['decade'] = (df['release_year'] // 10) * 10
                df['is_classic'] = (df['release_year'] < 2000).astype(int)
                df['is_recent_hit'] = (df['release_year'] >= 2020).astype(int)

            # 4. Track Name Features
            print(f"[4/7] Track name features ({name})...")
            if 'track_name' in df.columns:
                # Clean track name (remove remix, remastered, featuring info, dll)
                clean_name = df['track_name'].astype(str).str.lower()
                clean_name = clean_name.str.replace(r'[\(\[].*?[\)\]]', '', regex=True)
                clean_name = clean_name.str.split(' - feat.').str[0]
                clean_name = clean_name.str.split(' - with').str[0]
                clean_name = clean_name.str.split(' - sped up').str[0]
                clean_name = clean_name.str.split(' - remastered').str[0]
                clean_name = clean_name.str.split(' - from').str[0]
                clean_name = clean_name.str.strip()

                df['track_name_length'] = clean_name.str.len()
                df['track_name_word_count'] = clean_name.str.count(' ') + 1

            # 5. Interaction Features
            print(f"[5/7] Interaction features ({name})...")
            if 'artist_avg_pop' in df.columns and 'danceability' in df.columns:
                df['artist_x_dance'] = df['artist_avg_pop'] * df['danceability']
                df['artist_x_energy'] = df['artist_avg_pop'] * df['energy']

        print("✓ Feature engineering completed!")
        return self

    def process_lyrics(self, n_components=20):
        """Process lyrics menggunakan NLP (Natural Language Processing)"""
        print("\n" + "=" * 80)
        print("📝 PROCESSING LYRICS (NLP)")
        print("=" * 80)

        if 'lyrics' not in self.train_df.columns:
            print("⚠ No lyrics column found, skipping NLP features")
            return self

        print(f"Extracting TF-IDF features (n_components={n_components})...")

        # Fill missing lyrics dengan empty string
        self.train_df['lyrics'] = self.train_df['lyrics'].fillna('')
        self.test_df['lyrics'] = self.test_df['lyrics'].fillna('')

        # TF-IDF Vectorization
        tfidf = TfidfVectorizer(
            max_features=500,
            min_df=5,
            max_df=0.8,
            ngram_range=(1, 2),
            stop_words='english'
        )

        train_tfidf = tfidf.fit_transform(self.train_df['lyrics'])
        test_tfidf = tfidf.transform(self.test_df['lyrics'])

        # Dimensionality Reduction dengan SVD
        svd = TruncatedSVD(n_components=n_components, random_state=42)
        train_lyrics_features = svd.fit_transform(train_tfidf)
        test_lyrics_features = svd.transform(test_tfidf)

        explained_variance = svd.explained_variance_ratio_.sum()
        print(f"✓ Explained variance: {explained_variance:.2%}")

        # Tambahkan ke dataframe
        lyrics_cols = [f'lyrics_feature_{i}' for i in range(n_components)]
        train_lyrics_df = pd.DataFrame(train_lyrics_features, columns=lyrics_cols, index=self.train_df.index)
        test_lyrics_df = pd.DataFrame(test_lyrics_features, columns=lyrics_cols, index=self.test_df.index)

        self.train_df = pd.concat([self.train_df, train_lyrics_df], axis=1)
        self.test_df = pd.concat([self.test_df, test_lyrics_df], axis=1)

        print(f"✓ Added {n_components} lyrics features")
        return self

    def prepare_features(self):
        """Prepare final feature set untuk modeling"""
        print("\n" + "=" * 80)
        print("🎯 PREPARING FEATURES FOR MODELING")
        print("=" * 80)

        # Ambil semua fitur numerik
        numeric_features = self.train_df.select_dtypes(include=[np.number]).columns.tolist()

        # Pastikan 'explicit' treated as numeric
        if 'explicit' in self.train_df.columns:
            self.train_df['explicit'] = self.train_df['explicit'].astype(int)
            self.test_df['explicit'] = self.test_df['explicit'].astype(int)
            if 'explicit' not in numeric_features:
                numeric_features.append('explicit')

        # Exclude kolom yang tidak digunakan untuk modeling
        exclude_cols = ['popularity', 'track_id', 'track_name', 'artists', 'lyrics', 'release_year']
        numeric_features = [f for f in numeric_features if f not in exclude_cols]

        # Fitur kategorik
        categorical_features = ['track_genre', 'key_mode', 'tempo_category', 'decade']
        self.categorical_features_raw = [f for f in categorical_features if f in self.train_df.columns]

        # Encode fitur kategorik
        encoded_cat_features = []
        print("Encoding categorical features...")
        for col in self.categorical_features_raw:
            le = LabelEncoder()
            combined_series = pd.concat([
                self.train_df[col].astype(str),
                self.test_df[col].astype(str)
            ])
            le.fit(combined_series)

            self.train_df[col + '_encoded'] = le.transform(self.train_df[col].astype(str))
            self.test_df[col + '_encoded'] = le.transform(self.test_df[col].astype(str))

            self.label_encoders[col] = le
            encoded_cat_features.append(col + '_encoded')

        self.categorical_features_encoded = encoded_cat_features
        self.features = numeric_features + encoded_cat_features

        print(f"✓ Total features for modeling: {len(self.features)}")

        # Impute missing values dengan median
        print("Applying imputation (median)...")
        self.imputer = SimpleImputer(strategy='median')
        self.train_df[self.features] = self.imputer.fit_transform(self.train_df[self.features])
        self.test_df[self.features] = self.imputer.transform(self.test_df[self.features])

        # Set categorical dtype untuk LightGBM (lebih efisien)
        for col in self.categorical_features_encoded:
            self.train_df[col] = self.train_df[col].astype('category')
            self.test_df[col] = self.test_df[col].astype('category')

        print("✓ Features prepared!")
        return self

    def train_models(self, cv_folds=5):
        """Train LightGBM model dengan cross-validation"""
        print("\n" + "=" * 80)
        print("🤖 TRAINING MODEL (LightGBM)")
        print("=" * 80)

        X = self.train_df[self.features]
        y = self.train_df['popularity']

        # Inisialisasi LightGBM Regressor
        lgbm = LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.01,
            num_leaves=31,
            max_depth=6,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

        # Cross-validation
        print(f"Training with {cv_folds}-fold cross-validation...")
        kfold = KFold(n_splits=cv_folds, shuffle=True, random_state=42)

        cv_scores = cross_val_score(
            lgbm, X, y,
            cv=kfold,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1
        )
        self.cv_scores = -cv_scores

        print(f"\nCross-Validation Results:")
        for i, score in enumerate(self.cv_scores, 1):
            print(f"  Fold {i}: RMSE = {score:.4f}")

        print(f"\n✓ Mean RMSE: {self.cv_scores.mean():.4f} (+/- {self.cv_scores.std():.4f})")

        # Train pada full dataset
        print("\nTraining on full dataset...")
        lgbm.fit(X, y)
        self.models['LightGBM'] = lgbm

        # Get Out-of-Fold predictions untuk analisis
        print("\nGetting OOF predictions for analysis...")
        self.oof_predictions = cross_val_predict(lgbm, X, y, cv=kfold, n_jobs=-1)

        # Save results
        self.model_results = pd.DataFrame({
            'Model': ['LightGBM'],
            'Mean RMSE': [self.cv_scores.mean()],
            'Std RMSE': [self.cv_scores.std()]
        })

        return self

    def create_submission(self, predictions, filename='submission.csv'):
        """Create submission file untuk kompetisi"""
        print("\n" + "=" * 80)
        print(f"📤 CREATING SUBMISSION: {filename}")
        print("=" * 80)

        submission = pd.DataFrame({
            'track_id': self.test_df['track_id'],
            'popularity': predictions
        })

        # Clip predictions ke range [0, 100]
        submission['popularity'] = np.clip(submission['popularity'], 0, 100)

        output_path = self.output_path / filename
        submission.to_csv(output_path, index=False)

        print(f"✓ Saved: {output_path}")
        print(f"  • Predictions: {len(submission)}")
        print(f"  • Range: [{submission['popularity'].min():.2f}, {submission['popularity'].max():.2f}]")
        print(f"  • Mean:  {submission['popularity'].mean():.2f}")
        print(f"  • Std:   {submission['popularity'].std():.2f}")

        return submission

print("✅ SongPopularityPredictor class defined successfully!")

## 🚀 3. Initialize Predictor

Buat instance dari SongPopularityPredictor dengan path ke data Anda.

In [ ]:
# Konfigurasi paths
# Sesuaikan data_path dengan lokasi file train.csv dan test.csv
predictor = SongPopularityPredictor(
    data_path='.',  # Current directory (ubah sesuai lokasi data Anda)
    output_path='./outputs'
)

print("✅ Predictor initialized!")

## 📂 4. Load Data

Load training dan testing data dari CSV files.

In [ ]:
predictor.load_data()

### Preview Data

Lihat sample dari training data.

In [ ]:
# Preview training data
print("Training Data Preview:")
display(predictor.train_df.head())

print("\nTraining Data Info:")
predictor.train_df.info()

## 📊 5. Exploratory Data Analysis (EDA)

Analisis statistik dasar untuk memahami data.

In [ ]:
predictor.eda()

### Visualisasi Target Distribution

In [ ]:
# Visualize target distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(predictor.train_df['popularity'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Popularity')
plt.ylabel('Frequency')
plt.title('Popularity Distribution')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(predictor.train_df['popularity'])
plt.ylabel('Popularity')
plt.title('Popularity Boxplot')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Correlation Heatmap (Audio Features)

In [ ]:
# Audio features correlation
audio_features = ['danceability', 'energy', 'loudness', 'speechiness', 
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 
                  'tempo', 'popularity']

plt.figure(figsize=(12, 10))
corr = predictor.train_df[audio_features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Audio Features Correlation with Popularity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 6. Feature Engineering

Membuat fitur-fitur baru yang informatif:
- Artist features (target encoding)
- Audio interactions
- Temporal features
- Track name features
- Interaction features

In [ ]:
predictor.engineer_features()

### Preview Engineered Features

In [ ]:
# Lihat fitur yang sudah di-engineer
new_features = ['artist_avg_pop', 'artist_song_count', 'energy_x_dance', 
                'duration_min', 'years_since_release', 'decade', 
                'is_classic', 'is_recent_hit', 'track_name_length', 
                'track_name_word_count', 'artist_x_dance', 'artist_x_energy']

print("Engineered Features Preview:")
display(predictor.train_df[new_features].head())

print("\nEngineered Features Statistics:")
display(predictor.train_df[new_features].describe())

## 📝 7. Process Lyrics (NLP)

Menggunakan TF-IDF dan SVD untuk extract semantic features dari lyrics.

In [ ]:
predictor.process_lyrics(n_components=20)

## 🎯 8. Prepare Features for Modeling

Encoding, imputation, dan persiapan final features.

In [ ]:
predictor.prepare_features()

print(f"\n✅ Total features ready for modeling: {len(predictor.features)}")
print(f"\nFeature list:")
for i, feat in enumerate(predictor.features, 1):
    print(f"{i:3d}. {feat}")

## 🤖 9. Train Model

Train LightGBM dengan 5-Fold Cross-Validation.

In [ ]:
predictor.train_models(cv_folds=5)

### Feature Importance Analysis

In [ ]:
# Get feature importance
feature_importance = predictor.models['LightGBM'].feature_importances_
fi_df = pd.DataFrame({
    'Feature': predictor.features,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

# Top 20 features
plt.figure(figsize=(12, 8))
top_20 = fi_df.head(20)
colors = plt.cm.viridis(np.linspace(0, 1, len(top_20)))
plt.barh(range(len(top_20)), top_20['Importance'], color=colors, edgecolor='black')
plt.yticks(range(len(top_20)), top_20['Feature'])
plt.xlabel('Importance', fontweight='bold')
plt.title('Top 20 Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 20 Most Important Features:")
display(top_20)

### Model Performance Analysis

In [ ]:
# Calculate metrics
y = predictor.train_df['popularity']
oof_rmse = np.sqrt(mean_squared_error(y, predictor.oof_predictions))
oof_mae = mean_absolute_error(y, predictor.oof_predictions)
oof_r2 = r2_score(y, predictor.oof_predictions)

print("="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"\nOut-of-Fold Metrics:")
print(f"  RMSE: {oof_rmse:.4f}")
print(f"  MAE:  {oof_mae:.4f}")
print(f"  R²:   {oof_r2:.4f}")

# Actual vs Predicted plot
plt.figure(figsize=(10, 6))
plt.scatter(y, predictor.oof_predictions, alpha=0.3, s=10, c=y, cmap='viridis')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Popularity', fontweight='bold')
plt.ylabel('Predicted Popularity', fontweight='bold')
plt.title('Actual vs Predicted (Out-of-Fold)', fontsize=14, fontweight='bold')
plt.colorbar(label='Actual Popularity')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Residuals Analysis

In [ ]:
# Residuals
residuals = y - predictor.oof_predictions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals distribution
axes[0].hist(residuals, bins=60, edgecolor='black', alpha=0.7, color='coral')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0].set_xlabel('Residual', fontweight='bold')
axes[0].set_ylabel('Frequency', fontweight='bold')
axes[0].set_title('Residuals Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residual plot
axes[1].scatter(predictor.oof_predictions, residuals, alpha=0.3, s=10, c=y, cmap='coolwarm')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Popularity', fontweight='bold')
axes[1].set_ylabel('Residual', fontweight='bold')
axes[1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residual Statistics:")
print(f"  Mean:     {residuals.mean():.4f}")
print(f"  Std:      {residuals.std():.4f}")
print(f"  Skewness: {stats.skew(residuals):.4f}")
print(f"  Kurtosis: {stats.kurtosis(residuals):.4f}")

## 📤 10. Create Submission

Predict pada test set dan create submission file.

In [ ]:
# Predict on test set
X_test = predictor.test_df[predictor.features]
predictions = predictor.models['LightGBM'].predict(X_test)

# Create submission
submission = predictor.create_submission(predictions, 'submission_siklus4_enhanced.csv')

# Preview submission
print("\nSubmission Preview:")
display(submission.head(10))

# Submission statistics
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(submission['popularity'], bins=50, edgecolor='black', alpha=0.7, color='green')
plt.xlabel('Predicted Popularity')
plt.ylabel('Frequency')
plt.title('Test Set Predictions Distribution')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(predictor.train_df['popularity'], bins=50, alpha=0.6, label='Train', edgecolor='black')
plt.hist(submission['popularity'], bins=50, alpha=0.6, label='Test Predictions', edgecolor='black')
plt.xlabel('Popularity')
plt.ylabel('Frequency')
plt.title('Train vs Test Predictions')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 11. Final Summary

In [ ]:
print("\n" + "="*80)
print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)

print(f"\n🎯 FINAL METRICS:")
print(f"  OOF RMSE: {oof_rmse:.4f}")
print(f"  CV Mean:  {predictor.cv_scores.mean():.4f}")
print(f"  CV Std:   {predictor.cv_scores.std():.4f}")

print(f"\n📊 FEATURES:")
print(f"  Total features: {len(predictor.features)}")
print(f"  Top feature: {fi_df.iloc[0]['Feature']} (importance: {fi_df.iloc[0]['Importance']:.0f})")

print(f"\n📁 OUTPUT FILES:")
print(f"  • submission_siklus4_enhanced.csv")

if oof_rmse < 16.10:
    print(f"\n🏆 EXCELLENT! Competitive performance!")
elif oof_rmse < 16.30:
    print(f"\n🎉 VERY GOOD! Strong baseline!")
else:
    print(f"\n💪 GOOD! Ready for next iteration!")

print("\n" + "="*80)

## 📚 Next Steps

1. Review feature importance untuk understand model
2. Analyze error patterns di worst predictions
3. Consider improvement:
   - Hyperparameter tuning
   - Ensemble methods
   - Additional features
4. Submit predictions dan monitor leaderboard score

---

**Happy Modeling! 🎵🚀**